# Murphy 1 · Chapter 11 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M11). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 11.1 · Lasso by coordinate descent  
*Section §11.4 · code*

Implement lasso with coordinate descent and soft-thresholding on synthetic data with 10 features where only 3 truly matter. Print the weights for λ = 0.01, 0.1, 0.5 and count how many are exactly zero.

<details><summary><b>Hint</b></summary>

For feature j: ρ = xⱼᵀ(y − Xw + xⱼwⱼ)/N; wⱼ = soft(ρ, λ) / (xⱼᵀxⱼ/N), with soft(ρ, λ) = sign(ρ)·max(|ρ| − λ, 0).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(2)
N, D = 200, 10
X = rng.normal(size=(N, D)); w_true = np.zeros(D); w_true[[0, 3, 7]] = [3, -2, 1.5]
y = X @ w_true + 0.5 * rng.normal(size=N)
soft = lambda r, lam: np.sign(r) * np.maximum(np.abs(r) - lam, 0)
for lam in [0.01, 0.1, 0.5]:
    w = np.zeros(D)
    for sweep in range(200):
        for j in range(D):
            r = y - X @ w + X[:, j] * w[j]                 # residual without feature j
            rho = X[:, j] @ r / N
            w[j] = ...  # TODO: soft-threshold, then rescale
    print(f"lambda {lam}: zeros {np.sum(w == 0)}  w = {w.round(2)}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(2)
N, D = 200, 10
X = rng.normal(size=(N, D)); w_true = np.zeros(D); w_true[[0, 3, 7]] = [3, -2, 1.5]
y = X @ w_true + 0.5 * rng.normal(size=N)
soft = lambda r, lam: np.sign(r) * np.maximum(np.abs(r) - lam, 0)
for lam in [0.01, 0.1, 0.5]:
    w = np.zeros(D)
    for sweep in range(200):
        for j in range(D):
            r = y - X @ w + X[:, j] * w[j]                 # residual without feature j
            rho = X[:, j] @ r / N
            w[j] = soft(rho, lam) / (X[:, j] @ X[:, j] / N)  # soft-threshold, then rescale
    print(f"lambda {lam}: zeros {np.sum(w == 0)}  w = {w.round(2)}")

<details><summary><b>Answer and explanation</b></summary>

As λ grows, more weights hit exactly 0, and the three true features (with weights 3, −2, 1.5) survive longest. Soft-thresholding is the reason: any coefficient whose correlation with the residual is below λ gets set to exactly 0, not just shrunk (Murphy §11.4.3, §11.4.9).

Expected output:
```
lambda 0.01: zeros 2  w = [ 3.07  0.    0.03 -2.07  0.03 -0.01  0.03  1.43 -0.02  0.  ]
lambda 0.1: zeros 7  w = [ 2.98 -0.    0.   -1.99  0.   -0.    0.    1.33 -0.    0.  ]
lambda 0.5: zeros 7  w = [ 2.55 -0.    0.   -1.59  0.   -0.    0.    0.9  -0.    0.  ]
```

</details>

## 11.2 · What ridge does to each direction  
*Section §11.3 · derive*

Using the SVD X = USVᵀ, show that ridge predictions are ŷ = Σⱼ uⱼ · sⱼ²/(sⱼ² + λ) · uⱼᵀy. Which directions get shrunk most?

<details><summary><b>Hint</b></summary>

ŵ = (XᵀX + λI)⁻¹Xᵀy. Substitute XᵀX = VS²Vᵀ.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

(VS²Vᵀ + λI)⁻¹ = V(S² + λI)⁻¹Vᵀ, so ŵ = V(S² + λI)⁻¹SUᵀy and ŷ = Xŵ = US(S² + λI)⁻¹SUᵀy = Σ uⱼ [sⱼ²/(sⱼ²+λ)] uⱼᵀy. Least squares keeps every direction fully (factor 1). Ridge multiplies each by sⱼ²/(sⱼ²+λ): directions with large singular values (where the data varies a lot) barely shrink, while small-variance directions, the noisy ones, shrink toward 0. That's 'soft PCA' (Murphy §11.3.2).

</details>

## 11.3 · Outliers: squared error vs Huber  
*Section §11.6 · code*

Fit a line to 50 points where 5 are wild outliers, once with least squares and once with the Huber loss (δ = 1) by gradient descent. Compare the slopes with the true slope 2.

<details><summary><b>Hint</b></summary>

Huber gradient on residual r is r if |r| ≤ δ, else δ·sign(r).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(3)
x = rng.uniform(0, 10, 50); y = 2 * x + 1 + rng.normal(0, 1, 50)
y[:5] += rng.uniform(25, 40, 5)                       # five outliers
A = np.column_stack([np.ones(50), x])
w_ls = np.linalg.lstsq(A, y, rcond=None)[0]
w = w_ls.copy()
for _ in range(5000):
    r = A @ w - y
    g = ...  # TODO: Huber: linear pull beyond delta = 1
    w -= 0.01 * A.T @ g / 50
print("least squares slope", w_ls[1].round(3), " intercept", w_ls[0].round(3))
print("Huber slope        ", w[1].round(3), " intercept", w[0].round(3))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(3)
x = rng.uniform(0, 10, 50); y = 2 * x + 1 + rng.normal(0, 1, 50)
y[:5] += rng.uniform(25, 40, 5)                       # five outliers
A = np.column_stack([np.ones(50), x])
w_ls = np.linalg.lstsq(A, y, rcond=None)[0]
w = w_ls.copy()
for _ in range(5000):
    r = A @ w - y
    g = np.where(np.abs(r) <= 1, r, np.sign(r))  # Huber: linear pull beyond delta = 1
    w -= 0.01 * A.T @ g / 50
print("least squares slope", w_ls[1].round(3), " intercept", w_ls[0].round(3))
print("Huber slope        ", w[1].round(3), " intercept", w[0].round(3))

<details><summary><b>Answer and explanation</b></summary>

Least squares gets dragged toward the outliers: squaring a residual of 30 gives it 900 units of pull. Huber treats big residuals linearly, so each outlier's pull is capped at δ and the slope stays close to 2. Equivalent view: squared error assumes Gaussian noise, which says outliers are nearly impossible; Huber and Laplace assume heavier tails (Murphy §11.6).

Expected output:
```
least squares slope 1.217  intercept 7.954
Huber slope         1.87  intercept 1.721
```

</details>

## 11.4 · R² and what it means  
*Section §11.2 · code*

Fit least squares to y = 3x + noise with noise σ = 1, 3, 10 (200 points, x ~ U(0, 1)·5). Print R² for each. Then explain why R² falls even though the fitted slope is still right.

<details><summary><b>Hint</b></summary>

R² = 1 − SS_res / SS_tot.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(4)
x = rng.uniform(0, 5, 200); A = np.column_stack([np.ones(200), x])
for s in [1, 3, 10]:
    y = 3 * x + rng.normal(0, s, 200)
    w = np.linalg.lstsq(A, y, rcond=None)[0]
    r2 = ...  # TODO: share of variance explained
    print(f"noise {s:2d}: slope {w[1]:.3f}  R^2 {r2:.3f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(4)
x = rng.uniform(0, 5, 200); A = np.column_stack([np.ones(200), x])
for s in [1, 3, 10]:
    y = 3 * x + rng.normal(0, s, 200)
    w = np.linalg.lstsq(A, y, rcond=None)[0]
    r2 = 1 - np.sum((y - A @ w) ** 2) / np.sum((y - y.mean()) ** 2)  # share of variance explained
    print(f"noise {s:2d}: slope {w[1]:.3f}  R^2 {r2:.3f}")

<details><summary><b>Answer and explanation</b></summary>

The slope stays near 3 in all three cases (3.10, 2.91, 3.07), but R² drops from 0.95 to 0.66 to 0.15. R² measures the share of y's variation that x explains. With more noise there's more unexplained variation, even though the model is exactly right. A low R² means a noisy target, not necessarily a wrong model (Murphy §11.2.4).

Expected output:
```
noise  1: slope 3.100  R^2 0.948
noise  3: slope 2.910  R^2 0.661
noise 10: slope 3.069  R^2 0.149
```

</details>